In [ ]:
from pathlib import Path
import os
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "auctions_final.csv").exists():
    raise RuntimeError("Open Jupyter from the project folder (see START_HERE.md).")


In [ ]:
import pandas as pd

In [ ]:
auction = pd.read_csv('auction_data.csv')

In [ ]:
deliveries = pd.read_csv('deliveries.csv')

In [ ]:
matches = pd.read_csv('matches.csv')

In [ ]:
auction.head()

In [ ]:
auction.name.nunique()

In [ ]:
import pandas as pd

# Load auction + deliveries
auctions = pd.read_csv("auction_data.csv")
deliveries = pd.read_csv("deliveries.csv")

delivery_names = set(deliveries["batsman"].unique()) | set(deliveries["bowler"].unique())
delivery_names

In [ ]:

def match_to_deliveries(name):
    parts = name.strip().split()
    if len(parts) == 1:
        return name  # rare single-name case
    first, last = parts[0], parts[-1]
    candidates = [d for d in delivery_names if d.lower().endswith(last)]
    print(name, candidates)

    if len(candidates) == 0:
        return None
    
    if len(candidates) == 1:
        return candidates[0]
    else:
        # Check initials
        for i in range(len(first)):
            first_initial = first[i].upper()
            for c in candidates:
                if c.startswith(first_initial):
                    return c
            # fallback: return raw last name (debug)
        return None

auctions["matched_name"] = auctions["name"].apply(match_to_deliveries)

# Save mapping for inspection
auctions[["name", "matched_name"]].to_csv("auction_name_to_deliveries.csv", index=False)


In [ ]:
auctions["matched_name"]

In [ ]:
import pandas as pd

names = pd.read_csv("auction_name_to_deliveries.csv")

In [ ]:
names.isna().sum()

In [ ]:
len(names)

In [ ]:
131/1183

In [ ]:
names = names.dropna()

In [ ]:
names.to_csv("auction_name_to_deliveries.csv")

In [ ]:
auction = pd.read_csv("auction_data.csv")

In [ ]:
len(auction)

In [ ]:
len(names)

In [ ]:
names = names.drop_duplicates(subset=["name"])


In [ ]:
merged = auction.merge(names, how="left", on="name")

In [ ]:
len(merged)

In [ ]:
cleaned = merged.dropna(subset=["matched_name"])

In [ ]:
len(cleaned)

In [ ]:
cleaned["name"] = cleaned["matched_name"]

# Drop the extra column (optional)
cleaned = cleaned.drop(columns=["matched_name"])

# Save the cleaned dataset
cleaned.to_csv("auction_data_cleaned.csv", index=False)

In [ ]:
deliveries.groupby("batsman")

In [ ]:
deliveries

In [ ]:
deliveries1 = deliveries.merge(matches, how="left", on="match_id")

In [ ]:
deliveries1

In [ ]:
import pandas as pd

batsmen = pd.read_csv("auction roi/batsmen_price.csv")
bowlers = pd.read_csv("auction roi/bowlers_price_clean.csv")
allrounders = pd.read_csv("auction roi/allrounders_price.csv")

print(batsmen.head())
print(bowlers.head())
print(allrounders.head())


In [ ]:
# Correlation between price and runs
print("Batsmen correlation:", batsmen[["price", "runs"]].corr().iloc[0,1])

# Correlation between price and wickets
print("Bowlers correlation:", bowlers[["price", "wickets"]].corr().iloc[0,1])

# Correlation between price and allrounder score
print("Allrounders correlation:", allrounders[["price", "allrounder_score"]].corr().iloc[0,1])


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Batsmen
plt.figure(figsize=(8,5))
sns.regplot(data=batsmen, x="price", y="runs", scatter_kws={'alpha':0.5})
plt.title("Auction Price vs Runs (Batsmen)")
plt.xlabel("Price (INR)")
plt.ylabel("Runs")
plt.show()

# Bowlers
plt.figure(figsize=(8,5))
sns.regplot(data=bowlers, x="price", y="wickets", scatter_kws={'alpha':0.5})
plt.title("Auction Price vs Wickets (Bowlers)")
plt.xlabel("Price (INR)")
plt.ylabel("Wickets")
plt.show()

# Allrounders
plt.figure(figsize=(8,5))
sns.regplot(data=allrounders, x="price", y="allrounder_score", scatter_kws={'alpha':0.5})
plt.title("Auction Price vs Allrounder Score")
plt.xlabel("Price (INR)")
plt.ylabel("Allrounder Score")
plt.show()



In [ ]:
# 📌 IPL ROI Analysis Plots
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
batsmen = pd.read_csv("auction roi/batsmen_roi_inflation.csv")
bowlers = pd.read_csv("auction roi/bowlers_roi_inflation.csv")
allrounders = pd.read_csv("auction roi/allrounders_roi_inflation.csv")

# Set style
sns.set(style="whitegrid", context="talk")


In [ ]:
# Scatter: price vs runs
plt.figure(figsize=(10,6))
sns.scatterplot(data=batsmen, x="adj_price", y="runs", hue="season", alpha=0.7)
sns.regplot(data=batsmen, x="adj_price", y="runs", scatter=False, color="black")
plt.title("Batsmen: Auction Price vs Runs")
plt.xlabel("Adjusted Price (₹, 2025 terms)")
plt.ylabel("Runs")
plt.show()

# # Bar: Top 5 batsmen per season
# top_batsmen = batsmen.groupby("season").apply(lambda x: x.nlargest(5, "runs")).reset_index(drop=True)
# plt.figure(figsize=(12,6))
# sns.barplot(data=top_batsmen, x="season", y="runs", hue="player")
# plt.title("Top 5 Batsmen per Season (with Adjusted Price)")
# plt.ylabel("Runs")
# plt.show()


In [ ]:
# Scatter: price vs wickets
plt.figure(figsize=(10,6))
sns.scatterplot(data=bowlers, x="adj_price", y="wickets", hue="season", alpha=0.7)
sns.regplot(data=bowlers, x="adj_price", y="wickets", scatter=False, color="black")
plt.title("Bowlers: Auction Price vs Wickets")
plt.xlabel("Adjusted Price (₹, 2025 terms)")
plt.ylabel("Wickets")
plt.show()

# Boxplot: price per wicket distribution
plt.figure(figsize=(12,6))
sns.boxplot(data=bowlers, x="season", y="price_per_wicket_adj")
plt.title("Distribution of Price per Wicket (Inflation Adjusted)")
plt.ylabel("Price per Wicket (₹)")
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Bubble plot: runs vs wickets, bubble size = adj_price
plt.figure(figsize=(12,8))
sns.scatterplot(
    data=allrounders, x="runs", y="wickets", size="adj_price",
    hue="season", sizes=(50,500), alpha=0.6
)
plt.title("Allrounders: Runs vs Wickets (Bubble = Adjusted Price)")
plt.xlabel("Runs")
plt.ylabel("Wickets")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# Bar: best ROI allrounder per season
best_allrounders = allrounders.loc[allrounders.groupby("season")["price_per_allrounder_score_adj"].idxmin()]
plt.figure(figsize=(12,6))
sns.barplot(data=best_allrounders, x="season", y="price_per_allrounder_score_adj", hue="player")
plt.title("Best Allrounder ROI per Season (Lower is Better)")
plt.ylabel("Price per Allrounder Score (₹)")
plt.show()


In [ ]:
# Avg price_per_run trend
batsmen_trend = batsmen.groupby("season")["price_per_run_adj"].mean().reset_index()
plt.figure(figsize=(10,6))
sns.lineplot(data=batsmen_trend, x="season", y="price_per_run_adj", marker="o")
plt.title("Trend of Price per Run (2008-2025)")
plt.ylabel("Avg Price per Run (₹)")
plt.show()

# Avg price_per_wicket trend
bowlers_trend = bowlers.groupby("season")["price_per_wicket_adj"].mean().reset_index()
plt.figure(figsize=(10,6))
sns.lineplot(data=bowlers_trend, x="season", y="price_per_wicket_adj", marker="o", color="red")
plt.title("Trend of Price per Wicket (2008-2025)")
plt.ylabel("Avg Price per Wicket (₹)")
plt.show()


In [ ]:
# statsmodels is installed through requirements.txt

In [ ]:
# Correlation heatmap (Batsmen)
plt.figure(figsize=(6,5))
sns.heatmap(batsmen[["adj_price","runs","price_per_run_adj"]].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation: Price vs Performance (Batsmen)")
plt.show()

# Regression (quick check)
import statsmodels.api as sm
X = batsmen["adj_price"]
y = batsmen["runs"]
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
import statsmodels.api as sm
import numpy as np

# Polynomial fit (quadratic)
X = np.column_stack((batsmen["adj_price"], batsmen["adj_price"]**2))
X = sm.add_constant(X)
model_poly = sm.OLS(batsmen["runs"], X).fit()
print(model_poly.summary())


In [ ]:
intervals = [[1,2],[2,30],[3,4],[1,3]]

In [ ]:
intervals.sort(key=lambda x: x[1])
intervals

In [ ]:
del(intervals[1])

In [ ]:
intervals

In [ ]:
a = 10
print(~a)

In [ ]:
print(bin(a))

In [ ]:
print(bin(~a))